# EMG Signal Analysis

**Pipeline** (applied to every channel in the selected folder):
1. Raw signal
2. → Bandpass filter (6–500 Hz, 4th-order Butterworth, zero-phase `filtfilt`)
3. → Feedforward comb filter: `y[n] = x[n] − x[n−M]`, `M = round(fs/50)` — notches at 50, 100, 150 … Hz

**Output — 3 interactive Plotly figures:**
- **Fig 1 — Time domain:** rows = channels, cols = Raw | Bandpass | Comb
- **Fig 2 — FFT:** same grid, frequency domain
- **Fig 3 — FFT overlay:** all 3 stages overlaid per channel with 50 Hz harmonic markers

Run cells top to bottom, then use the widget in the last cell.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import signal as sig

import plotly.graph_objects as go
from plotly.subplots import make_subplots

import ipywidgets as widgets
from IPython.display import display, clear_output

RECORDINGS_DIR = Path("recordings")

In [ ]:
# ── data ─────────────────────────────────────────────────────────────────────

def load_signal(folder: str, channel: str):
    path = RECORDINGS_DIR / folder / f"{channel}.csv"
    df = pd.read_csv(path)
    t = df["time"].values
    x = df["data"].values
    fs = round(1.0 / (t[1] - t[0]))
    return t, x, float(fs)

# ── filters ───────────────────────────────────────────────────────────────────

def bandpass_filter(x: np.ndarray, fs: float,
                    low_hz: float = 6.0, high_hz: float = 500.0,
                    order: int = 4) -> np.ndarray:
    """Zero-phase Butterworth bandpass.
    Uses SOS form — b/a form is numerically unstable at low cutoffs vs fs."""
    sos = sig.butter(order, [low_hz, high_hz], btype="band", fs=fs, output="sos")
    return sig.sosfiltfilt(sos, x)

def comb_filter(x: np.ndarray, fs: float, f0: float = 50.0) -> np.ndarray:
    """Feedforward comb filter: y[n] = x[n] - x[n-M], M = round(fs / f0).
    Subtracts a one-period-delayed copy → nulls at f0, 2f0, 3f0 … up to Nyquist.
    First M samples are left unfiltered (one-period warm-up)."""
    M = int(round(fs / f0))
    y = x.copy()
    y[M:] -= x[:-M]
    return y

# ── FFT ───────────────────────────────────────────────────────────────────────

def rfft_single_sided(x: np.ndarray, fs: float):
    N = len(x)
    freqs = np.fft.rfftfreq(N, d=1.0 / fs)
    mag = np.abs(np.fft.rfft(x)) * (2.0 / N)
    mag[0] /= 2.0
    if N % 2 == 0:
        mag[-1] /= 2.0
    return freqs, mag

# ── palette ───────────────────────────────────────────────────────────────────

STAGE_STYLES = [
    # (label, line_color, fill_color, time_key, fft_key)
    ("Raw",               "#1565C0", "rgba(21,101,192,0.10)",  "raw", "fft_raw"),
    ("Bandpass 6–500 Hz", "#E65100", "rgba(230,81,0,0.10)",    "bp",  "fft_bp"),
    ("Bandpass + Comb",   "#2E7D32", "rgba(46,125,50,0.10)",   "cmb", "fft_cmb"),
]

CHANNELS = [
    ("emg_line_L", "Line In — Left"),
    ("emg_line_R", "Line In — Right"),
    ("emg_mic",    "Microphone"),
]

# ── main ──────────────────────────────────────────────────────────────────────

def analyze_folder(folder: str, fft_max_hz: float = 1000.0):
    """Process every channel in *folder*; one figure set per channel."""

    chs = []
    for ch_key, ch_label in CHANNELS:
        path = RECORDINGS_DIR / folder / f"{ch_key}.csv"
        if not path.exists():
            print(f"  Skipping {ch_label}: {ch_key}.csv not found")
            continue
        print(f"Loading {ch_key}.csv …", end="  ")
        t, raw, fs = load_signal(folder, ch_key)
        print(f"fs={fs:.0f} Hz  {len(raw):,} samples  {t[-1]:.3f} s")

        bp  = bandpass_filter(raw, fs)
        cmb = comb_filter(bp, fs)
        fr, fft_raw = rfft_single_sided(raw, fs)
        _,  fft_bp  = rfft_single_sided(bp,  fs)
        _,  fft_cmb = rfft_single_sided(cmb, fs)

        chs.append(dict(
            label=ch_label,
            time=t, raw=raw, bp=bp, cmb=cmb,
            fr=fr, fft_raw=fft_raw, fft_bp=fft_bp, fft_cmb=fft_cmb,
        ))

    if not chs:
        print("No channel files found in", folder)
        return

    n_harm = min(int(fft_max_hz / 50), 20)

    for cd in chs:
        fr = cd["fr"]
        fm = fr <= fft_max_hz

        # ══════════════════════════════════════════════════════════════════════
        # Figure A — 3 rows (stages) × 2 cols (time | FFT)
        # ══════════════════════════════════════════════════════════════════════
        fig_a = make_subplots(
            rows=3, cols=2,
            subplot_titles=[
                "Raw — time",               "Raw — FFT",
                "Bandpass 6–500 Hz — time", "Bandpass 6–500 Hz — FFT",
                "Bandpass + Comb — time",   "Bandpass + Comb — FFT",
            ],
            column_widths=[0.55, 0.45],
            vertical_spacing=0.10,
            horizontal_spacing=0.08,
        )

        for r, (sl, color, fill, sig_key, fft_key) in enumerate(STAGE_STYLES, 1):
            fig_a.add_trace(go.Scatter(
                x=cd["time"], y=cd[sig_key],
                name=sl, line=dict(width=0.6, color=color),
                showlegend=False,
            ), row=r, col=1)
            fig_a.add_trace(go.Scatter(
                x=fr[fm], y=cd[fft_key][fm],
                name=sl, line=dict(width=1.0, color=color),
                fill="tozeroy", fillcolor=fill,
                showlegend=False,
            ), row=r, col=2)
            fig_a.update_xaxes(title_text="Time (s)",       row=r, col=1)
            fig_a.update_xaxes(title_text="Frequency (Hz)", row=r, col=2)
            fig_a.update_yaxes(title_text="Amplitude (V)",  row=r, col=1)
            fig_a.update_yaxes(title_text="Magnitude (V)",  row=r, col=2)

        fig_a.update_layout(
            title_text=f"{cd['label']} — {folder}",
            height=700,
            template="plotly_white",
        )
        fig_a.show()

        # ══════════════════════════════════════════════════════════════════════
        # Figure B — FFT overlay + 50 Hz harmonic markers
        # ══════════════════════════════════════════════════════════════════════
        fig_b = go.Figure()
        for sl, color, _, _, fft_key in STAGE_STYLES:
            fig_b.add_trace(go.Scatter(
                x=fr[fm], y=cd[fft_key][fm],
                name=sl, line=dict(width=1.2, color=color),
            ))

        for k in range(1, n_harm + 1):
            fig_b.add_vline(
                x=k * 50,
                line=dict(color="rgba(180,0,0,0.25)", width=1, dash="dot"),
                annotation_text=str(k * 50),
                annotation_position="top right",
                annotation_font_size=8,
                annotation_font_color="rgba(180,0,0,0.55)",
            )

        fig_b.update_layout(
            title_text=f"{cd['label']} — FFT Overlay — {folder}",
            xaxis_title="Frequency (Hz)",
            yaxis_title="Magnitude (V)",
            height=380,
            template="plotly_white",
            legend=dict(orientation="h", y=1.12, x=0.5, xanchor="center"),
        )
        fig_b.show()

    print("Done.")


In [ ]:
# Clear any stale widget output left by previous runs of this cell.
# VSCode Jupyter appends cell output instead of replacing it, so without
# this call re-running the cell stacks new widgets on top of old ones,
# leaving multiple live buttons in the kernel — each with its own handler.
clear_output(wait=True)

folders = sorted([f.name for f in RECORDINGS_DIR.iterdir() if f.is_dir()])

if not folders:
    print("No recording folders found in 'recordings/'. Run dual_acquisition.py first.")
else:
    # Singleton pattern via try/except — more reliable than globals() in VSCode Jupyter.
    # On first run: create all widgets.  On re-run: just refresh the folder list.
    try:
        _emg_run_btn
    except NameError:
        _emg_folder_w = widgets.Dropdown(
            options=folders,
            value=folders[-1],
            description="Recording:",
            style={"description_width": "initial"},
            layout=widgets.Layout(width="440px"),
        )
        _emg_fft_w = widgets.BoundedIntText(
            value=1000, min=100, max=24000, step=50,
            description="FFT max Hz:",
            style={"description_width": "initial"},
            layout=widgets.Layout(width="220px"),
        )
        _emg_run_btn = widgets.Button(
            description="▶  Analyze & Plot",
            button_style="primary",
            layout=widgets.Layout(width="190px", height="38px"),
        )
        _emg_out = widgets.Output()
    else:
        _emg_folder_w.options = folders
        _emg_folder_w.value = folders[-1]

    def _on_run(b):
        with _emg_out:
            clear_output(wait=True)
            analyze_folder(_emg_folder_w.value, fft_max_hz=float(_emg_fft_w.value))

    # [:] = [] is a guaranteed in-place replace of the list contents —
    # eliminates every handler accumulated from previous runs before adding one.
    _emg_run_btn._click_handlers.callbacks[:] = []
    _emg_run_btn.on_click(_on_run)

    display(widgets.VBox([
        widgets.HTML("<b>Select a recording, then click Analyze & Plot.<br>"
                     "All channels (Line L, Line R, Mic) are plotted automatically.</b>"),
        _emg_folder_w,
        _emg_fft_w,
        _emg_run_btn,
        _emg_out,
    ]))
